[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/snowflake/snowflake.ipynb)

# Snowflake Growth

This example computes a snowflake growing: a seed of ice in undercooled water growing into a six-armed dendrite. Water cooled below its melting point freezes onto any ice it touches, and freezing releases latent heat, which warms the water at the ice's edge and slows the freezing there. A bump on the edge reaches into colder water and sheds its heat to more sides than the flat edge around it, so it grows faster: the tips outrun the sides, and each tip sprouts branches of its own. The ice's surface energy depends on how its edge lies against the crystal's axes, six-fold for ice, and steers the fastest tips along six directions, the six arms of a snowflake. The ice is a phase field, the fraction of ice at each place, one in ice and nothing in water, with a thin, smooth edge between them.

The algebra is the plane joined with three directions that have no metric, for the volume, the ice and the heat a parcel of water holds, `Algebra("x+y+") * Algebra("v0i0h0")`. A parcel is a point weighed by its volume and offset along the other two by the ice and the heat it holds, which `mv.i &` and `mv.h &` read off it. Freezing is one fixed direction among these amounts, `freezing = ice + latent_heat * heat`, ice gained and its latent heat released, and the plane that reads a parcel's energy, `mv.h - latent_heat * mv.i`, pairs with it to nothing: freezing only moves energy from the latent store into the heat. A gradient wedged with the amount it acts on is a drive, which a material map takes to the flux it drives; the heat's drive is conducted, and the ice's drive paired with its own flux, `drive & spreading(drive)`, is the squared steepness of the ice's edge, with no metric square taken. The edge's width varies with its normal through the turn from the crystal's axis to the steepness, `turn = steepness * crystal_axis`, a scaled rotor whose sixth power has the six-fold `cos(6 angle)` as its scalar part, with no angle taken. The derivative of such an anisotropic gradient energy is notoriously fiddly by hand; here it is written out, the six-fold term's slope being the rotor's fifth power, `6 * crystal_axis * turn ** 5`, and each vertex's share its hat's drive paired with the flux that slope drives. JAX only compiles the stepping.

In the vector-calculus notation of phase-field models, with $\phi$ the ice fraction, $T$ the temperature and $\theta$ the angle of the edge's normal $\nabla\phi$ from the crystal's axis, the free energy is $F = \int \tfrac12\,\epsilon(\theta)^2\,|\nabla\phi|^2 + f(\phi, T)\,dA$, with the width $\epsilon(\theta) = \bar\epsilon\,(1 + \delta\cos 6\theta)$ and the tilted double well $f = \tfrac14\phi^2(1-\phi)^2 - \tfrac16\,m(T)\,\phi^2(3 - 2\phi)$, $m(T) = \tfrac{\alpha}{\pi}\arctan\gamma(T_m - T)$. The ice relaxes as $\tau\,\partial_t\phi = -\delta F/\delta\phi = \nabla\cdot(\epsilon^2\nabla\phi) - \partial_x(\epsilon\,\epsilon'\,\partial_y\phi) + \partial_y(\epsilon\,\epsilon'\,\partial_x\phi) + \phi(1-\phi)(\phi - \tfrac12 + m)$, and the heat as $\partial_t T = \nabla^2 T + K\,\partial_t\phi$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples,jax] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import jax
import numpy as np

from numga import Algebra, stack
from numga.backend.jax import JaxContext
from numga.sparse import spdiag
from examples.util import shapes
from examples.util.animation import show_states
from examples.thermodynamics.snowflake import render, scenarios

# The plane, and the volume, the ice and the heat a parcel of water holds: each a direction of its own, degenerate,
# since nothing measures a volume, an amount of ice or of heat against a length.
ga = Algebra("x+y+") * Algebra("v0i0h0")
mv = JaxContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
# An amount a cell holds, intrinsic like the join that measures its area: the area, or the energy in it.
Amount = ga.gatype.antipseudoscalar()                                         # Antipseudoscalar-
# The planes reading a place's run across and up; the drives, a gradient wedged with the ice or the heat.
Upright = ga.gatype.from_blades("x y")                                        # Vector+
Drive = ga.gatype.from_blades("xi yi xh yh")                                  # Bivector+
# The plane's two directions, across and up.
directions = stack((mv.x.dual(), mv.y.dual()))                                # [2] Point-
# A unit of volume, the weight of a place, a unit of ice and a unit of heat.
volume = mv.v.dual()                                                          # [] Point-
ice = mv.i.dual()                                                             # [] Point-
heat = mv.h.dual()                                                            # [] Point-

## The dish of water

The water fills a round dish, meshed in nearly equilateral triangles laid in staggered rings around its middle. The mesh has no six-fold symmetry of its own; a lattice of equilateral triangles would have one, and would carry its six directions into the crystal's growth. The water starts cold, at temperature zero, with a seed of ice in its middle. Temperatures are measured from the cold water's in units of its undercooling, so that ice melts at one, and times are in units in which heat diffuses across a unit of length in a unit of time. A field is interpolated across each triangle by its corners' hats, whose gradients are planes built from the triangle's sides by joins alone. The triangle's area stays intrinsic, like the join of its sides that measures it, and so does what a cell holds: an energy's change at a vertex is an amount, and only over the cell's area does it become a rate.

In [ ]:
# The dish, its vertices places of unit volume.
topology, coordinates = shapes.concentric_disk(np.array([scenarios.dish_radius]), scenarios.n_divisions)
vertices = (volume + mv(Upright, coordinates).dual()).field()                 # Point-[vertices]
# The water at the start: unit volume, the seed's ice, and no heat yet, its type holding the heat's direction for the
# run to fill.
parcels = (volume + ice * scenarios.seed_fractions(coordinates) + 0 * heat).field()   # Point-[vertices]

sides = topology.boundary[2, 1] * (topology.d[0] * vertices)                  # [corners] Point-[faces]
# Twice a triangle's area: two of its sides joined, at unit volume, ice and heat.
double_areas = (((sides[0] & sides[1]) & volume) & ice) & heat                # Scalar-[faces]
areas = double_areas / 2                                                      # Scalar-[faces]
# A corner's hat gradient: the plane through the facing side, along volume, ice and heat, over twice the area, which
# reads one from the side to the corner.
gradients = (volume & ice & heat & sides) / double_areas                      # [corners] Plane+[faces]
# Each vertex's share of the area around it.
corners = topology.incidence[2, 0]                                            # [corners] Scalar+[faces <- vertices]
cells = (~corners * (areas / 3)).sum(axis=0)                                  # Scalar-[vertices]
render.draw_seed(topology, vertices, parcels);

The seed close up in the dish's triangles: a disc of ice, pale, a fifth of a unit across, in the cold water, dark. The innermost ring of the mesh is a hexagon, and each ring outward holds a few more vertices than the one inside it, so the rings soon fall out of step with any six-fold pattern.

## The ice's free energy

The ice fraction relaxes down a free energy summed over the dish: a double well with its minima at water and ice, tilted toward ice by the undercooling, and a gradient energy, `areas * widths ** 2 * (drive & spreading(drive)) / 2` over the triangles, which makes the edge between them cost energy and sets its width. The gradient energy pairs the ice's drive, minus the steepness of its fraction wedged with the ice, with the flux the drive pushes, and so measures the squared steepness without a metric square; the same material map conducts the heat. The edge's width varies six-fold with its normal, through the sixth power of the turn from the crystal's axis to the steepness, so that the ice's surface costs a little more where its normal lies along one of the crystal's six axes.

In [ ]:
# The ice's relaxation time, its edge's mean width, the six-fold anisotropy of that width, and the steepness below
# which the anisotropy fades, where the ice fraction is flat.
relaxation_time, mean_width, anisotropy, regularization = 3e-4, 0.01, 0.02, 1.0
# The undercooling's tilt of the double well, its strength and its steepness, and the melting temperature.
tilt_strength, tilt_steepness, melting_temperature = 0.9, 10.0, 1.0
# The crystal's axis, a plane whose normal, up, one arm grows along.
crystal_axis = mv.y                                                           # [] Plane+

In [ ]:
# A field's gradient across each triangle: its corners' values on their hats' gradients.
gradient = (spdiag(gradients) * corners).sum(axis=0) * Scalar                 # (Plane+ <- Scalar+)[faces <- vertices]
# The material map, the same for the heat and the ice at unit rate: each of the plane's directions swept through the
# amount, read by a drive and driving the amount along it.
sweeps = stack((directions & heat, directions & ice))                         # [amounts, 2] Flux-
spreading = (sweeps * (sweeps & Drive)).sum(axis=-1).sum(axis=0)             # [] Flux- <- Drive+

The free energy's slope with respect to each vertex's ice fraction drives the freezing, and it is written out. The six-fold term's slope with respect to the steepness is the turn's fifth power, `6 * crystal_axis * turn ** 5`: the derivative of the six-fold factor extended off the unit normals as a homogeneous polynomial, as the Cahn–Hoffman vector of materials science is the derivative of a surface energy extended so. Each vertex's share of the gradient energy's change is its hat's drive paired with the energy's slope, a flux: twice the ice's own flux, `spreading(drive)`, is the slope of its squared steepness.

The six-fold term is the one place where the plane's own metric enters. The squared steepness pairs a drive with its flux and needs none, but the angle between the edge's normal and the crystal's axis does: the turn `steepness * crystal_axis` is a geometric product in `x+y+`, as it should be, since the crystal's six directions are a fact about angles in its lattice. Its slope therefore comes out as a plane, measured by that metric, where the gradient energy's other slope is a flux, and the material map turns it into one, `spreading(-(slope ^ mv.i))`. That is right because the spreading here is the plane's metric itself, the same in every direction; with a gradient energy that measured steepness unevenly, the turn would have to be measured in that energy's metric too.

In [ ]:
# Each corner's hat's drive on the ice.
ice_drives = -(gradients ^ mv.i)                                              # [corners] Drive+[faces]


def free_energy_slope(fraction: Scalar, temperatures: Scalar, anisotropy: float) -> Amount:
    """The derivative of the ice's free energy with respect to its fraction at each vertex `Scalar-[vertices]`, at this
    ice fraction and these temperatures `Scalar+[vertices]`, with this anisotropy."""
    steepness = gradient(fraction)                                            # Plane+[faces]
    # The ice's drive and the flux it drives; paired, the edge's squared steepness, whose slope is twice the flux.
    drive = -(steepness ^ mv.i)                                               # Drive+[faces]
    flux = spreading(drive)                                                   # Flux-[faces]
    squared_steepness = drive & flux                                          # Scalar+[faces]
    turn = steepness * crystal_axis                                           # Even+[faces]
    third_power = turn ** 3                                                   # Even+[faces]
    six_fold = third_power.scalar_product(third_power)                        # Scalar+[faces]
    reach = squared_steepness + regularization ** 2                           # Scalar+[faces]
    widths = mean_width * (1 + anisotropy * six_fold / reach ** 3)            # Scalar+[faces]
    # The six-fold term's slope with respect to the steepness, the turn's fifth power: a plane, measured in the plane's
    # metric, and the flux the material map, that same metric here, makes of it.
    slope = 6 * crystal_axis * turn ** 5                                      # Plane+[faces]
    slope_flux = spreading(-(slope ^ mv.i))                                   # Flux-[faces]
    # The widths' change as the edge turns; the gradient energy's slope, the widths squared times the flux and that
    # change times the steepness squared.
    turning = mean_width * anisotropy * (slope_flux / reach ** 3 - 6 * six_fold * flux / reach ** 4)   # Flux-[faces]
    pull = widths ** 2 * flux + squared_steepness * widths * turning          # Flux-[faces]
    # Each vertex's share of the gradient energy's change: its hat's drive paired with the energy's slope.
    shares = (~corners * (areas * (ice_drives & pull))).sum(axis=0)           # Scalar-[vertices]
    # The double well's slope, tilted toward ice by the undercooling.
    tilt = tilt_strength / np.pi * (tilt_steepness * (melting_temperature - temperatures)).arctan()   # Scalar+[vertices]
    well_slope = fraction * (1 - fraction) * (1 - 2 * fraction) / 2 - tilt * fraction * (1 - fraction)   # Scalar+[vertices]
    return shares + cells * well_slope

## Growing the snowflake

Each step, every parcel moves along freezing by as much ice as the free energy's slope drives over the relaxation time, so the heat it gains is the latent heat of the ice it gained, and the heat is conducted: each hat's drive on the heat, through the same material map, is read by each other hat for the heat it carries out of that hat's cell. The rim lets no heat through. JAX compiles the step and scans the steps between frames. The cells below take about ten seconds, most of it stepping the water.

In [ ]:
# The latent heat's warming per unit of ice; the time step, the frames and the steps between them, and the frames'
# duration in milliseconds.
latent_heat = 1.8
dt, n_frames, stride = 2e-4, 100, 14
frame_duration_ms = 40

# Freezing: ice gained, and its latent heat released.
freezing = ice + latent_heat * heat                                           # [] Point-
# The heat each vertex's cell loses to each vertex's warmth: each hat's drive on the heat conducted, and read by the
# other hats for its heat, over the triangles.
heat_drives = -(gradients ^ mv.h)                                             # [corners] Drive+[faces]
local = areas * (mv.h & (gradients[:, None] ^ spreading(heat_drives[None, :])))   # [corners, corners] Scalar-[faces]
conduction = (~corners[:, None] * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)   # Scalar-[vertices <- vertices]

In [ ]:
def growth(parcels: Point, anisotropy: float) -> Iterator[Point]:
    """The water's parcels `Point-[vertices]` at each frame, from these, the ice's edge's width varied six-fold by this
    anisotropy."""

    def step(parcels: Point, _) -> tuple[Point, None]:
        # The ice relaxes down its free energy, moving each parcel along freezing, and the heat is conducted.
        fraction, temperatures = mv.i & parcels, mv.h & parcels               # Scalar+[vertices], Scalar+[vertices]
        frozen = -free_energy_slope(fraction, temperatures, anisotropy) / (relaxation_time * cells)   # Scalar+[vertices]
        conducted = -(conduction * temperatures) / cells                      # Scalar+[vertices]
        return parcels + dt * (frozen * freezing + conducted * heat), None

    # The steps between two frames, compiled and scanned.
    @jax.jit
    def strided(parcels: Point) -> Point:
        return jax.lax.scan(step, parcels, None, length=stride)[0]

    for _ in range(n_frames):
        yield parcels
        parcels = strided(parcels)


snowflake = show_states(growth(parcels, anisotropy), render.animate, topology, vertices, name="snowflake",
                        duration_ms=frame_duration_ms)

The seed grows first into a hexagonal plate, its corners on the crystal's six axes and its faces turned between them, where the edge costs least. The corners reach into colder water and grow into arms, and each arm's sides break into branches, which grow along the crystal's axes too. The pale halo around the ice is the latent heat its freezing released, spreading into the cold water, dark. The finished crystal is held a moment before the loop starts over.

## Without the crystal's preference

The same seed in the same water grows again with the anisotropy taken away, its edge's width the same whichever way the edge faces. This cell takes about ten seconds, most of it stepping the water.

In [ ]:
isotropic = show_states(growth(parcels, 0.0), render.animate, topology, vertices, name="snowflake_isotropic",
                        duration_ms=frame_duration_ms)

Without the crystal's preference the edge is as unstable as before: its bumps still outrun their sides and split. But no direction is favoured, and the seed grows into a round bush of fingers, their number and their directions set by small irregularities, the mesh's among them, with no six-fold order; where neighbouring fingers met, they trapped drops of water in a ring around the middle. The snowflake's six arms come from the crystal's anisotropy, not from the mesh.

In [ ]:
# checks
# Freezing moves energy from the latent store into the heat: the energy plane pairs with it to nothing.
energy = mv.h - latent_heat * mv.i                                            # [] Plane+
np.testing.assert_allclose((energy & freezing).to_array(), 0.0, atol=1e-15)
# The rim holds the heat in, so in both runs the water's energy summed over the cells stays what it was: the heat
# gained is the latent heat of the ice gained.
for states in (snowflake, isotropic):
    first, last = ((cells * (energy & state)).sites.sum().to_array() for state in (states[0], states[-1]))
    np.testing.assert_allclose(last, first, rtol=1e-10)
# The seed's plate turns its faces between the crystal's axes, while without the anisotropy the seed's edge faces every
# way alike: the edge's mean six-fold cos(6 angle) from the crystal's axis, weighted by its squared steepness, at the
# frame where the plate shows.
plate_frame = 8
alignments = []
for states in (snowflake, isotropic):
    steepness = gradient(mv.i & states[plate_frame])                         # Plane+[faces]
    squared_steepness = -(steepness ^ mv.i) & spreading(-(steepness ^ mv.i))  # Scalar+[faces]
    third_power = (steepness * crystal_axis) ** 3                             # Even+[faces]
    six_fold = third_power.scalar_product(third_power)                        # Scalar+[faces]
    alignments.append(((areas * six_fold / (squared_steepness + regularization ** 2) ** 2).sites.sum()
                       / (areas * squared_steepness).sites.sum()).to_array())
assert alignments[0] < -0.4
assert abs(alignments[1]) < 0.1